<a href="https://colab.research.google.com/github/Danial-071/Langgraph-project/blob/main/ReflectionAgentWorkflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install langgraph
!pip install langchain-openrouter
!pip install langchain-core
!pip install langchain-classic
!pip install langchain-community
!pip install pygraphviz

In [ ]:
from langchain_core.prompts.chat import MessagesPlaceholder
from langchain_openrouter import ChatOpenRouter
from langgraph.graph import MessageGraph
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from typing import Sequence, TypedDict, List, Sequence
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from google.colab import userdata
import os

In [ ]:
os.environ['OPENROUTER_API_KEY'] = userdata.get('API')

llm = ChatOpenRouter(model = userdata.get('model'))

In [ ]:
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a professional LinkedIn content assistant tasked with crafting engaging, insightful, and well-structured LinkedIn posts."
            " Generate the best LinkedIn post possible for the user's request."
            " If the user provides feedback or critique, respond with a refined version of your previous attempts, improving clarity, tone, or engagement as needed.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [ ]:
generate_chain = generation_prompt | llm

In [ ]:
reflection_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a professional LinkedIn content strategist and thought leadership expert. Your task is to critically evaluate the given LinkedIn post and provide a comprehensive critique. Follow these guidelines:

        1. Assess the post’s overall quality, professionalism, and alignment with LinkedIn best practices.
        2. Evaluate the structure, tone, clarity, and readability of the post.
        3. Analyze the post’s potential for engagement (likes, comments, shares) and its effectiveness in building professional credibility.
        4. Consider the post’s relevance to the author’s industry, audience, or current trends.
        5. Examine the use of formatting (e.g., line breaks, bullet points), hashtags, mentions, and media (if any).
        6. Evaluate the effectiveness of any call-to-action or takeaway.

        Provide a detailed critique that includes:
        - A brief explanation of the post’s strengths and weaknesses.
        - Specific areas that could be improved.
        - Actionable suggestions for enhancing clarity, engagement, and professionalism.

        Your critique will be used to improve the post in the next revision step, so ensure your feedback is thoughtful, constructive, and practical.
        """
    ),
    MessagesPlaceholder(variable_name="messages")
])

In [ ]:
reflect_chain = reflection_prompt | llm

In [ ]:
graph = MessageGraph()
def generation_node(state: Sequence[BaseMessage]) -> List[BaseMessage]:
    generated_post = generate_chain.invoke({"messages": state})
    return [AIMessage(content=generated_post.content)]

/tmp/ipykernel_1638/3413922384.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = MessageGraph()


In [ ]:
def reflection_node(messages: Sequence[BaseMessage]) -> List[BaseMessage]:
    res = reflect_chain.invoke({"messages": messages})  # Passes messages as input to reflect_chain
    return [HumanMessage(content=res.content)]  # Returns the refined message as HumanMessage for feedback

In [ ]:
def should_continue(state: List[BaseMessage]):
    print(state)
    print(len(state))
    print("----------------------------------------------------------------------")
    if len(state) > 6:
        return END
    return "reflect"

In [ ]:
graph.add_node("generate", generation_node)
graph.add_node("reflect", reflection_node)
graph.add_edge("reflect", "generate")
graph.add_conditional_edges("generate", should_continue)
graph.set_entry_point("generate")
workflow = graph.compile()
inputs = HumanMessage(content="""Write a linkedin post on getting a software developer job at IBM under 160 characters""")
response = workflow.invoke(inputs)

[HumanMessage(content='Write a linkedin post on getting a software developer job at IBM under 160 characters', additional_kwargs={}, response_metadata={}, id='ac434fe0-2615-4283-81a8-fde8817290ef'), AIMessage(content='Landed my dream role as a Software Developer at IBM! 🚀 Grateful for the journey and excited to build the future of tech. #IBM #SoftwareEngineer #NewJob #TechCareers', additional_kwargs={}, response_metadata={}, id='5a0d03e5-8b70-4063-a334-ca6d64c14e36', tool_calls=[], invalid_tool_calls=[])]
2
----------------------------------------------------------------------


In [ ]:
print("First response : " ,response[1].content)
print("Second response : ", response[2].content)

In [ ]:
response[-1].content

In [ ]:
from IPython.display import Image, display

display(Image(workflow.get_graph().draw_png()))